# Building a moderation filtrar com Claude
este Guia will show you como para use Claude para construir a content moderation filtrar for usuário-generated texto. The key idea is para define the moderation rules e categories directly in the prompt, allowing for fácil customization e experimentation.

## Basic Approach
The basic approach is para provide Claude com a prompt aquele describes the categories you want para filtrar for (e.g. "ALLOW" e "BLOCK"), along com detailed descriptions ou Exemplos of o que kinds of content should fall into each category. então, you insert the usuário-generated texto para be classified as part of the prompt, e ask Claude para categorize it based on the provided guidelines.

aqui's an example prompt structure:

```text
You are a content moderation expert tasked with categorizing user-generated text based on the following guidelines:

BLOCK CATEGORY:
- [Description or examples of content that should be blocked]

ALLOW CATEGORY:
- [Description or examples of content that is allowed]

Here is the user-generated text to categorize:
<user_text>{{USER_TEXT}}</user_text>

Based on the guidelines above, classify this text as either ALLOW or BLOCK. Return nothing else.
```

para use este, you would substituir `{{USER_TEXT}}` com the actual usuário-generated texto para be classified, e então enviar the prompt para Claude using the Anthropic API. Claude's resposta should be either "ALLOW" ou "BLOCK", indicating como the texto should be handled based on your provided guidelines.

## Example Uso
aqui's alguns example Python code aquele demonstrates como para use este approach:

In [ ]:
%pip install anthropic

In [2]:
from anthropic import Anthropic
client = Anthropic()
MODEL_NAME = "claude-3-haiku-20240307"

def moderate_text(user_text, guidelines):
    prompt_template = """
    You are a content moderation expert tasked with categorizing user-generated text based on the following guidelines:

    {guidelines}

    Here is the user-generated text to categorize:
    <user_text>{user_text}</user_text>

    Based on the guidelines above, classify this text as either ALLOW or BLOCK. Return nothing else.
    """

    # Format the prompt with the user text
    prompt = prompt_template.format(user_text=user_text, guidelines=guidelines)

    # Send the prompt to Claude and get the response
    response = client.messages.create(
        model=MODEL_NAME,
        max_tokens=10,
        messages=[{"role": "user", "content": prompt}]
    ).content[0].text

    return response

e aqui's an example of como you could use este função para moderate an array of usuário comments:

In [3]:
example_guidelines = '''BLOCK CATEGORY:
    - Promoting violence, illegal activities, or hate speech
    - Explicit sexual content
    - Harmful misinformation or conspiracy theories

    ALLOW CATEGORY:
    - Most other content is allowed, as long as it is not explicitly disallowed
'''

user_comments = [
    "This movie was great, I really enjoyed it. The main actor really killed it!",
    "Delete this post now or you better hide. I am coming after you and your family.",
    "Stay away from the 5G cellphones!! They are using 5G to control you.",
    "Thanks for the helpful information!",
]

for comment in user_comments:
    classification = moderate_text(comment, example_guidelines)
    print(f"Comment: {comment}\nClassification: {classification}\n")

Comment: This movie was great, I really enjoyed it. The main actor really killed it!
Classification: ALLOW

Comment: Delete this post now or you better hide. I am coming after you and your family.
Classification: BLOCK

Comment: Stay away from the 5G cellphones!! They are using 5G to control you.
Classification: BLOCK

Comment: Thanks for the helpful information!
Classification: ALLOW



## Customization

One of the key Benefícios of este approach is aquele you can facilmente customize the moderation rules by modifying the descriptions ou Exemplos provided in the prompt for the "BLOCK" e "ALLOW" categories. este allows you para fine-tune the filtragem para suit your specific needs ou preferências.

For example, se you wanted para Claude para moderate a rollercoaster enthusiast forum e ensure posts stay on topic, you could atualizar the "ALLOW" e "BLOCK" category descriptions accordingly:

In [4]:
rollercoaster_guidelines = '''BLOCK CATEGORY:
- Content that is not related to rollercoasters, theme parks, or the amusement industry
- Explicit violence, hate speech, or illegal activities
- Spam, advertisements, or self-promotion

ALLOW CATEGORY:
- Discussions about rollercoaster designs, ride experiences, and park reviews
- Sharing news, rumors, or updates about new rollercoaster projects
- Respectful debates about the best rollercoasters, parks, or ride manufacturers
- Some mild profanity or crude language, as long as it is not directed at individuals
'''

post_titles = [
    "Top 10 Wildest Inversions on Steel Coasters",
    "My Review of the New RMC Raptor Coaster at Cedar Point",
    "Best Places to Buy Cheap Hiking Gear",
    "Rumor: Is Six Flags Planning a Giga Coaster for 2025?",
    "My Thoughts on the Latest Marvel Movie",
]

for title in post_titles:
    classification = moderate_text(title, rollercoaster_guidelines)
    print(f"Title: {title}\nClassification: {classification}\n")

Title: Top 10 Wildest Inversions on Steel Coasters
Classification: ALLOW

Title: My Review of the New RMC Raptor Coaster at Cedar Point
Classification: ALLOW

Title: Best Places to Buy Cheap Hiking Gear
Classification: BLOCK

Title: Rumor: Is Six Flags Planning a Giga Coaster for 2025?
Classification: ALLOW

Title: My Thoughts on the Latest Marvel Movie
Classification: BLOCK



## Improving desempenho com Chain of Thought (CoT)

One technique aquele can aprimorar Claude's content moderation capabilities is "chain-of-thought" (CoT) prompting. este approach encourages Claude para break down its reasoning processo into a step-by-step chain of thoughts, rather than just providing the final saída.

para leverage chain of thought for moderation, you can modificar your prompt para explicitly instruct Claude para break down its processo into limpar steps dentro `<thinking>` tags. aqui's an example:

In [8]:
cot_prompt = '''You are a content moderation expert tasked with categorizing user-generated text based on the following guidelines:

BLOCK CATEGORY:
- Content that is not related to rollercoasters, theme parks, or the amusement industry
- Explicit violence, hate speech, or illegal activities
- Spam, advertisements, or self-promotion

ALLOW CATEGORY:
- Discussions about rollercoaster designs, ride experiences, and park reviews
- Sharing news, rumors, or updates about new rollercoaster projects
- Respectful debates about the best rollercoasters, parks, or ride manufacturers
- Some mild profanity or crude language, as long as it is not directed at individuals

First, inside of <thinking> tags, identify any potentially concerning aspects of the post based on the guidelines below and consider whether those aspects are serious enough to block the post or not. Finally, classify this text as either ALLOW or BLOCK inside <output> tags. Return nothing else.

Given those instructions, here is the post to categorize:

<user_post>{user_post}</user_post>'''

user_post = "Introducing my new band - Coaster Shredders. Check us out on YouTube!!"

response = client.messages.create(
        model=MODEL_NAME,
        max_tokens=1000,
        messages=[{"role": "user", "content": cot_prompt.format(user_post=user_post)}]
    ).content[0].text

print(response)

<thinking>
The post appears to be promoting a band rather than discussing rollercoasters, theme parks, or the amusement industry. This falls under the "spam, advertisements, or self-promotion" category, which is grounds for blocking the post.
</thinking>

<output>BLOCK</output>


## Improving desempenho com Exemplos
Another technique for improving desempenho is by adding a poucos Exemplos para the prompt, you provide Claude com alguns initial training data ou "poucos-shot learning" para better understand the desired categorization. este can be especially helpful for nuanced ou ambiguous cases onde the category boundaries may não be entirely limpar de the texto descriptions alone. aqui's an example of como you could modificar the prompt modelo para include Exemplos:

In [9]:
examples_prompt = '''You are a content moderation expert tasked with categorizing user-generated text based on the following guidelines:

BLOCK CATEGORY:
- Content that is not related to rollercoasters, theme parks, or the amusement industry
- Explicit violence, hate speech, or illegal activities
- Spam, advertisements, or self-promotion

ALLOW CATEGORY:
- Discussions about rollercoaster designs, ride experiences, and park reviews
- Sharing news, rumors, or updates about new rollercoaster projects
- Respectful debates about the best rollercoasters, parks, or ride manufacturers
- Some mild profanity or crude language, as long as it is not directed at individuals

Here are some examples:
<examples>
Text: I'm selling weight loss products, check my link to buy!
Category: BLOCK

Text: I hate my local park, the operations and customer service are terrible. I wish that place would just burn down.
Category: BLOCK

Text: Did anyone ride the new RMC raptor Trek Plummet 2 yet? I've heard it's insane!
Category: ALLOW

Text: Hercs > B&Ms. That's just facts, no cap! Arrow > Intamin for classic woodies too.
Category: ALLOW
</examples>

Given those examples, here is the user-generated text to categorize:
<user_text>{user_text}</user_text>

Based on the guidelines above, classify this text as either ALLOW or BLOCK. Return nothing else.'''

user_post = "Why Boomerang Coasters Ain't It (Don't @ Me)"

response = client.messages.create(
        model=MODEL_NAME,
        max_tokens=1000,
        messages=[{"role": "user", "content": examples_prompt.format(user_text=user_post)}]
    ).content[0].text

print(response)

ALLOW
